## Connectivity - PFM result plots

- overview of group average DAN patches
    (from `nets_PFM/npc_net_ana.ipynb`)
- 4 example subjects (2 per group) with small R-parietal-lateral patches ?!
- NPC mask example to illustrate tha DAN is replaced by visual cortex in parietal cortex

In [1]:
import sys
import os.path as op
import os

import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '/home/ubuntu/git/parietal_patterns/nets_PFM')
from config import ATLAS_NETWORK_NAMES

ATLAS = 'caNets_DDnr'
NETWORK_NAMES = ATLAS_NETWORK_NAMES[ATLAS]
DAN_LABEL = 5  # 'Dorsal-attention' in this atlas

BIDS_ROOT_LARGE = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'
BIDS_ORIG = '/mnt_03/ds-dnumrisk'
PFM_ROOT = '/mnt_03/ds-dnumrisk/derivatives/pfm_fslr'
plot_folder = op.join(BIDS_ROOT_LARGE, 'paper_materials')
os.makedirs(plot_folder, exist_ok=True)

group_df = pd.read_csv(op.join(BIDS_ROOT_LARGE, 'group_assignment.csv')).set_index('subject')

# matches nets_PFM/npc_net_ana.ipynb's active SUBJECTS list: its docstring says sub-05 is
# excluded, but that removal line is actually commented out there -- kept consistent here
SUBJECTS = list(range(1, 67))

In [2]:
# house style + shared plotting helpers (see prep_results/plotting.py) -- same module
# used for Fig 1, so both figures share fonts/spines/panel-letter conventions
import seaborn as sns

from parietal_patterns.prep_results.plotting import (
    set_style, render_surf_panel, plot_group_comparison_panel,
    add_panel_letter, add_subpanel_label,
    GROUP_LABELS, GROUP_ORDER, GROUP_PALETTE,
)

set_style()
group_labels = GROUP_LABELS
group_palette = GROUP_PALETTE

In [3]:
# PFM/DAN results are native fsLR 32k (see nets_PFM/CLAUDE.md) -- kept in that space here
# rather than resampled to fsaverage5, to avoid distorting the small patches that are the
# point of this analysis. Panel layout (lateral view only, L/R side by side) still matches
# Fig 1's house style.
from neuromaps.datasets import fetch_atlas
from parietal_patterns.utils.surfaces import build_adj_and_coords

fslr = fetch_atlas('fsLR', '32k')
adj_L, coords_L = build_adj_and_coords(str(fslr['midthickness'].L))
adj_R, coords_R = build_adj_and_coords(str(fslr['midthickness'].R))

In [4]:
# ── DAN patches from the group-average atlas (same logic as nets_PFM/npc_net_ana.ipynb) ──
from parietal_patterns.utils.surfaces import centroid_to_lobe
from scipy.sparse.csgraph import connected_components
from collections import defaultdict

atlas_path = op.join(PFM_ROOT, 'atlases',
                      f'{ATLAS}-magjudge-task-average-from-fsav5_space-fsLR_den-32k_cortex.npz')
atlas_labels = np.load(atlas_path)['labels']   # (64984,)
atlas_map_L = atlas_labels[:32492]
atlas_map_R = atlas_labels[32492:]

atlas_ref_patches = []
dan_patches_map = np.zeros_like(atlas_labels)
n_comp_L = 0
for hemi, adj, coords, hemi_map in [
    ('L', adj_L, coords_L, atlas_map_L),
    ('R', adj_R, coords_R, atlas_map_R),
    ]:
    net_verts = np.where(hemi_map == DAN_LABEL)[0]
    sub_adj = adj[net_verts][:, net_verts]
    n_comp, comp_labels = connected_components(sub_adj, directed=False)
    if hemi == 'L':
        n_comp_L = n_comp
    for c in range(n_comp):
        verts = net_verts[comp_labels == c]
        centroid = coords[verts].mean(axis=0)
        verts_hemi = verts if hemi == 'L' else verts + len(atlas_map_L)
        dan_patch_number = c + 1 if hemi == 'L' else c + 1 + n_comp_L
        dan_patches_map[verts_hemi] = dan_patch_number
        atlas_ref_patches.append({
            'ref_patch': None,
            'hemi': hemi,
            'n_verts': len(verts),
            'centroid': centroid,
            '_lobe': centroid_to_lobe(centroid),
            'patch_num': dan_patch_number,
        })

# largest patch per (hemi, lobe) gets the clean name; smaller ones get -2, -3, ...
lobe_groups = defaultdict(list)
for i, p in enumerate(atlas_ref_patches):
    lobe_groups[(p['hemi'], p['_lobe'])].append(i)
for (hemi, lobe), indices in lobe_groups.items():
    by_size = sorted(indices, key=lambda i: atlas_ref_patches[i]['n_verts'], reverse=True)
    for rank, idx in enumerate(by_size):
        atlas_ref_patches[idx]['ref_patch'] = f"{hemi}_{lobe}" if rank == 0 else f"{hemi}_{lobe}-{rank + 1}"

ref_df = pd.DataFrame(atlas_ref_patches).drop(columns=['centroid', '_lobe'])
ref_df = ref_df[ref_df['n_verts'] >= 10]  # drop specks unlikely to be meaningful
ref_df.sort_values(['hemi', 'n_verts'], ascending=[True, False])

,ref_patch,hemi,n_verts,patch_num
2,L_parietal-lateral,L,2041,3
0,L_frontal-lateral,L,1524,1
4,L_temporal,L,350,5
1,L_frontal-medial-dorsal,L,154,2
3,L_frontal-insula,L,20,4
5,L_frontal-medial,L,17,6
11,R_parietal-lateral,R,2145,12
8,R_frontal-lateral,R,1654,9
13,R_temporal,R,290,14
9,R_frontal-medial-dorsal,R,157,10


In [5]:
# top 4 patches per hemisphere (parietal-lateral, frontal-lateral, frontal-medial-dorsal,
# temporal, ... -- whichever 4 are largest) get their own color; everything else stays gray
N_top_patches = 8
N_per_hemi = N_top_patches // 2
atlas_ref_patches_filtered = []
for hemi in ['L', 'R']:
    hemi_patches = sorted(
        [p for p in atlas_ref_patches if p['hemi'] == hemi],
        key=lambda p: p['n_verts'], reverse=True,
    )
    atlas_ref_patches_filtered.extend(hemi_patches[:N_per_hemi])

_dan_num_to_name = {p['patch_num']: p['ref_patch'] for p in atlas_ref_patches_filtered}
_all_names = sorted(set(p['ref_patch'] for p in atlas_ref_patches_filtered))
_name_to_idx = {n: i + 1 for i, n in enumerate(_all_names)}  # 0 = unassigned/background

import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

colors = ["#C5C6CE",
          "#222E71", "#26828E",
          "#35B779", "#B9FD25",
          "#F4D63F", "#F3A112",
          "#8A2219", "#8E44AD"]
cmap = mcolors.ListedColormap(colors)
vmin, vmax = -0.5, len(_all_names) + 0.5

color_map_patches = np.zeros(len(dan_patches_map), dtype=float)
for num, name in _dan_num_to_name.items():
    color_map_patches[dan_patches_map == num] = _name_to_idx[name]

In [ ]:
# Group-average DAN patches, lateral view only, L & R side by side (medial patches noted
# in the caption rather than shown -- see prep_results/plotting.py for render_surf_panel).
# Same (elev, azim) camera tuples as Fig 1's surface panels, for a matching viewing angle
# despite the different mesh (fsLR 32k here vs fsaverage5 there).
view_L = (30, 30 + (360 / 2))
view_R = (30, -30)

panel_imgs_dan = {
    'L': render_surf_panel(str(fslr['inflated'].L), str(fslr['sulc'].L),
                            color_map_patches[:32492], view_L, cmap, vmin, vmax,
                            darkness=1.0),
    'R': render_surf_panel(str(fslr['inflated'].R), str(fslr['sulc'].R),
                            color_map_patches[32492:], view_R, cmap, vmin, vmax,
                            darkness=1.0),
}
print({k: v.size for k, v in panel_imgs_dan.items()})  # (width, height) px -- sizes the figure below

fig, axes = plt.subplots(1, 2, figsize=(7.25, 2.9), constrained_layout=True)
for ax, hemi in zip(axes, ['L', 'R']):
    ax.imshow(panel_imgs_dan[hemi])
    ax.axis('off')
    ax.text(0.5, 1.0, hemi, transform=ax.transAxes, ha='center', va='bottom', fontsize=10)

handles = [mpatches.Patch(color=cmap(i + 1), label=name.replace('_', ' '))
           for i, name in enumerate(_all_names)]
fig.legend(handles=handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.1), frameon=False, fontsize=8)

fig.savefig(op.join(plot_folder, f'ch3_PFM_DANpatches_group_average-{ATLAS}_lateral.pdf'),
            bbox_inches='tight')
fig.savefig(op.join(plot_folder, f'ch3_PFM_DANpatches_group_average-{ATLAS}_lateral.svg'),
            bbox_inches='tight')
plt.show()